# ANIP - Tâche 1 : Reconnaissance Faciale
## Version V10 — Alignement MTCNN + Calibration FAR + Super-Embedding

### Améliorations par rapport à V9
1. **Alignement facial MTCNN** : détection + recadrage + alignement des yeux avant extraction
2. **Super-embedding 1024-dim** : concaténation FaceNet(512) + ArcFace(512) au lieu de moyenne des scores
3. **Calibration FAR** : seuil fixé pour garantir FAR ≤ 0.1% (au lieu de Youden)
4. **TTA 6 vues** : ajout d'une vue gaussienne floue pour robustesse aux basse résolution

### Pourquoi l'alignement change tout
FaceNet et ArcFace sont entraînés sur des visages alignés (yeux aux mêmes coordonnées px).
Sans alignement, le modèle compare des zones différentes → embeddings bruités → AUC ≈ 0.75.
Avec alignement → embeddings cohérents → AUC attendu ≈ 0.85+

## 1. Imports

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
import cv2
from PIL import Image
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
from facenet_pytorch import InceptionResnetV1, MTCNN

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report, precision_recall_curve

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## 2. Configuration

In [ ]:
NOTEBOOK_DIR = Path('.').resolve()
DATA_PATH = NOTEBOOK_DIR / 'anip-reconnaissance-faciale-estimation-ages-ocr' / 'dataset_tache_1' / 'dataset_tache_1'
if not DATA_PATH.exists():
    candidates = list(NOTEBOOK_DIR.rglob('dataset_tache_1'))
    DATA_PATH = candidates[-1] if candidates else (_ for _ in ()).throw(FileNotFoundError('dataset_tache_1 introuvable'))

TRAIN_PATH = DATA_PATH / 'train'
TEST_PATH  = DATA_PATH / 'test'

IMG_SIZE   = (160, 160)
BATCH_SIZE = 32   # réduit car MTCNN est plus lourd
SEED       = 42
VAL_SIZE   = 0.2

USE_ARCFACE = True
W_FACENET   = 0.4
W_ARCFACE   = 0.6

# Calibration sur FAR fixe (False Acceptance Rate)
TARGET_FAR  = 0.001   # on accepte au max 0.1% de faux positifs

# TTA : 6 vues
TTA_VIEWS = [
    {'flip': False, 'angle': 0.0,   'brightness': 0.0,  'blur': False},
    {'flip': True,  'angle': 0.0,   'brightness': 0.0,  'blur': False},
    {'flip': False, 'angle': 10.0,  'brightness': 0.0,  'blur': False},
    {'flip': False, 'angle': -10.0, 'brightness': 0.0,  'blur': False},
    {'flip': False, 'angle': 0.0,   'brightness': 20.0, 'blur': False},
    {'flip': False, 'angle': 0.0,   'brightness': 0.0,  'blur': True},   # vue floue
]
TTA_N_VIEWS = len(TTA_VIEWS)

PRIOR_POS_OVERRIDE = None
THRESHOLD_FLOOR    = 0.45

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f'Train      : {TRAIN_PATH}')
print(f'Test       : {TEST_PATH}')
print(f'TTA vues   : {TTA_N_VIEWS}')
print(f'USE_ARCFACE: {USE_ARCFACE}')
print(f'TARGET_FAR : {TARGET_FAR}')

## 3. Chargement des données

In [ ]:
def _list_valid_jpgs(folder):
    exts = {'.jpg', '.jpeg', '.JPG', '.JPEG'}
    return sorted(p for p in folder.iterdir()
                  if p.is_file() and p.suffix in exts and not p.name.startswith('.')), 0

def parse_train_filename(fp):
    parts = fp.stem.split('_')
    return int(parts[0]), int(parts[1])

def load_train_data():
    imgs, _ = _list_valid_jpgs(TRAIN_PATH)
    data = []
    for p in imgs:
        try:
            pid, pnum = parse_train_filename(p)
            data.append({'filepath': str(p), 'person_id': pid, 'photo_num': pnum})
        except: pass
    return pd.DataFrame(data).sort_values(['person_id','photo_num']).reset_index(drop=True)

df_train = load_train_data()
print(f'Images : {len(df_train)} | Personnes : {df_train["person_id"].nunique()}')

pids = np.array(sorted(df_train['person_id'].unique()))
train_ids, val_ids = train_test_split(pids, test_size=VAL_SIZE, random_state=SEED)
df_train_split = df_train[df_train['person_id'].isin(train_ids)].reset_index(drop=True)
df_val_split   = df_train[df_train['person_id'].isin(val_ids)].reset_index(drop=True)
print(f'Val : {len(val_ids)} identités | {len(df_val_split)} images')

test_imgs_list, _ = _list_valid_jpgs(TEST_PATH)
n_test = len(test_imgs_list)
total_test_pairs = n_test * (n_test - 1) // 2
n_identites_test_est = n_test // 2
prior_pos = PRIOR_POS_OVERRIDE if PRIOR_POS_OVERRIDE else n_identites_test_est / total_test_pairs
print(f'\nTest set : {n_test} images | {total_test_pairs:,} paires totales')
print(f'Prior positif estimé : {prior_pos:.6f}  ({100*prior_pos:.4f}%)')

## 4. Modèles + MTCNN

### Nouveau en V10 : MTCNN
MTCNN détecte le visage, aligne les points de repère (yeux, nez, bouche)
et recadre l'image pour que les visages soient toujours dans la même position.
C'est exactement ce que FaceNet et ArcFace attendent en entrée.

In [ ]:
# ── MTCNN (alignement facial) ────────────────────────────────────────────────
print('Chargement MTCNN...')
mtcnn = MTCNN(
    image_size=160,
    margin=20,          # marge autour du visage détecté
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    keep_all=False,     # garder uniquement le visage le plus probable
    device=DEVICE,
    post_process=False  # on veut les pixels bruts [0,255], pas normalisés
)
print('MTCNN chargé ✓')

# ── FaceNet VGGFace2 ─────────────────────────────────────────────────────────
print('Chargement FaceNet VGGFace2...')
facenet = InceptionResnetV1(pretrained='vggface2', classify=False).eval().to(DEVICE)
for p in facenet.parameters():
    p.requires_grad = False
print('FaceNet chargé ✓  (512-dim embeddings)')

# ── ArcFace via insightface ──────────────────────────────────────────────────
arcface = None
if USE_ARCFACE:
    try:
        from insightface.model_zoo import get_model as insightface_get_model
        print('\nChargement ArcFace R100 (buffalo_l)...')
        ctx_id = 0 if torch.cuda.is_available() else -1
        arcface = insightface_get_model('buffalo_l', download=True, download_zip=True)
        arcface.prepare(ctx_id=ctx_id)
        print(f'ArcFace R100 chargé ✓')
    except Exception as e:
        print(f'[WARN] ArcFace non disponible : {e}')
        arcface = None

W_FACENET_ACT = 1.0 if arcface is None else W_FACENET
W_ARCFACE_ACT = 0.0 if arcface is None else W_ARCFACE
print(f'\nEnsemble : FaceNet×{W_FACENET_ACT} + ArcFace×{W_ARCFACE_ACT}')

## 5. Prétraitement avec alignement MTCNN

### Logique V10
1. MTCNN détecte et aligne le visage → tensor [0,255]
2. Si aucun visage détecté → fallback cv2.resize classique
3. Normalisation spécifique à chaque modèle

In [ ]:
@torch.no_grad()
def align_face_mtcnn(img_path):
    """
    Détecte et aligne le visage avec MTCNN.
    Retourne un numpy array RGB [0,255] de taille IMG_SIZE, ou None si échec.
    """
    try:
        img_pil = Image.open(str(img_path)).convert('RGB')
        face_tensor = mtcnn(img_pil)   # tensor (3, 160, 160) ou None
        if face_tensor is None:
            return None
        # Convertir en numpy [0,255]
        face_np = face_tensor.permute(1, 2, 0).numpy()
        face_np = np.clip(face_np, 0, 255).astype('float32')
        return face_np  # RGB, float32, [0,255]
    except:
        return None


def load_face(img_path):
    """
    Charge un visage : MTCNN en priorité, fallback cv2 si non détecté.
    Retourne numpy RGB [0,255] float32.
    """
    face = align_face_mtcnn(img_path)
    if face is not None:
        return face
    # Fallback
    img = cv2.imread(str(img_path))
    if img is None:
        return np.zeros((*IMG_SIZE, 3), dtype='float32')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, IMG_SIZE).astype('float32')


def apply_augmentation(face_np, flip=False, angle=0.0, brightness=0.0, blur=False):
    """Applique les augmentations TTA sur une image RGB [0,255]."""
    img = face_np.copy()
    if flip:
        img = img[:, ::-1, :].copy()
    if angle != 0.0:
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_REFLECT_101)
    if brightness != 0.0:
        img = np.clip(img + brightness, 0.0, 255.0)
    if blur:
        img = cv2.GaussianBlur(img, (3, 3), 0)
    return img


def to_facenet_tensor(face_rgb_255):
    """Normalise pour FaceNet : [-1, 1], CHW."""
    img = (face_rgb_255 - 127.5) / 128.0
    return np.transpose(img, (2, 0, 1)).astype('float32')


def to_arcface_array(face_rgb_255):
    """Normalise pour ArcFace : [-1,1], BGR, 112x112."""
    face_bgr = face_rgb_255[:, :, ::-1].copy()
    face_bgr = cv2.resize(face_bgr, (112, 112))
    img = (face_bgr / 255.0 - 0.5) / 0.5
    return np.transpose(img, (2, 0, 1)).astype('float32')


print(f'{TTA_N_VIEWS} vues TTA configurées :')
for i, v in enumerate(TTA_VIEWS):
    print(f'  Vue {i+1} : flip={v["flip"]}  angle={v["angle"]:+.0f}°  brightness={v["brightness"]:+.0f}  blur={v["blur"]}')

## 6. Extraction des embeddings

### Nouveau en V10 : Super-embedding 1024-dim
Au lieu de calculer deux similarités (FaceNet + ArcFace) séparément puis de les moyenner,
on concatène les vecteurs : [FaceNet(512) | ArcFace(512)] → super-vecteur 1024-dim.
La similarité cosinus sur ce vecteur capte les deux représentations simultanément.

In [ ]:
@torch.no_grad()
def get_embeddings_v10(filepaths, batch_size=32):
    """
    Pour chaque image :
    1. Alignement MTCNN (fallback cv2 si échec)
    2. TTA N vues → moyenne → L2-norm
    3. Super-embedding : concat(FaceNet, ArcFace) si ArcFace dispo
    
    Retourne : embs (N, 512) si FaceNet seul | (N, 1024) si ensemble
    """
    facenet.eval()
    filepaths = list(filepaths)
    all_embs = []
    mtcnn_success = 0
    mtcnn_fallback = 0

    for i in tqdm(range(0, len(filepaths), batch_size), desc='Embeddings V10'):
        batch = filepaths[i:i + batch_size]

        # Charger les visages alignés (une fois par image, réutilisé pour toutes les vues)
        faces_raw = []
        for fp in batch:
            face = align_face_mtcnn(fp)
            if face is not None:
                mtcnn_success += 1
            else:
                mtcnn_fallback += 1
                img = cv2.imread(str(fp))
                if img is None:
                    face = np.zeros((*IMG_SIZE, 3), dtype='float32')
                else:
                    face = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    face = cv2.resize(face, IMG_SIZE).astype('float32')
            faces_raw.append(face)

        # ── FaceNet TTA ───────────────────────────────────────────────────
        view_fn = []
        for view_params in TTA_VIEWS:
            imgs = np.array([to_facenet_tensor(apply_augmentation(f, **view_params))
                             for f in faces_raw], dtype='float32')
            t = torch.tensor(imgs).to(DEVICE)
            embs = facenet(t).cpu().numpy()
            view_fn.append(embs)
        avg_fn = np.mean(view_fn, axis=0)
        avg_fn = avg_fn / np.linalg.norm(avg_fn, axis=1, keepdims=True).clip(min=1e-8)

        # ── ArcFace TTA ───────────────────────────────────────────────────
        if arcface is not None:
            view_arc = []
            for view_params in TTA_VIEWS:
                imgs = np.array([to_arcface_array(apply_augmentation(f, **view_params))
                                 for f in faces_raw], dtype='float32')
                embs = arcface.get_feat(imgs)
                norms = np.linalg.norm(embs, axis=1, keepdims=True).clip(min=1e-8)
                view_arc.append(embs / norms)
            avg_arc = np.mean(view_arc, axis=0)
            avg_arc = avg_arc / np.linalg.norm(avg_arc, axis=1, keepdims=True).clip(min=1e-8)

            # Super-embedding 1024-dim
            super_emb = np.concatenate([avg_fn, avg_arc], axis=1)   # (B, 1024)
            super_emb = super_emb / np.linalg.norm(super_emb, axis=1, keepdims=True).clip(min=1e-8)
            all_embs.append(super_emb)
        else:
            all_embs.append(avg_fn)

    embs = np.vstack(all_embs)
    total = mtcnn_success + mtcnn_fallback
    print(f'MTCNN aligné : {mtcnn_success}/{total} ({100*mtcnn_success/total:.1f}%)  |  Fallback : {mtcnn_fallback}')
    print(f'Embeddings shape : {embs.shape}  ({"1024-dim super" if arcface else "512-dim FaceNet"})')
    return embs


print('Fonctions extraction V10 configurées.')

## 7. Extraction embeddings validation

In [ ]:
print('Extraction embeddings validation...')
val_embs = get_embeddings_v10(df_val_split['filepath'].values, BATCH_SIZE)
val_fps  = list(df_val_split['filepath'].values)
emb_dict = dict(zip(val_fps, val_embs))

# Paires de validation équilibrées
def create_pairs_balanced(df, seed=SEED):
    rng = np.random.default_rng(seed)
    groups = {pid: grp['filepath'].values
              for pid, grp in df.groupby('person_id') if len(grp) >= 2}
    person_ids = np.array(list(groups.keys()))
    pairs, labels, seen = [], [], set()
    pos_cands = [(a, b) for imgs in groups.values() for a, b in combinations(imgs, 2)]
    rng.shuffle(pos_cands)
    for a, b in pos_cands:
        seen.add(tuple(sorted((a, b)))); pairs.append([a, b]); labels.append(1)
    n_neg = len(labels)
    attempts = 0
    while sum(1 for l in labels if l == 0) < n_neg and attempts < n_neg * 30:
        attempts += 1
        p1, p2 = rng.choice(person_ids, 2, replace=False)
        i1, i2 = rng.choice(groups[p1]), rng.choice(groups[p2])
        key = tuple(sorted((i1, i2)))
        if key not in seen:
            seen.add(key); pairs.append([i1, i2]); labels.append(0)
    print(f'  Positifs : {sum(labels)}  |  Négatifs : {sum(1 for l in labels if l==0)}')
    return np.array(pairs), np.array(labels)

print('\nCréation des paires de validation...')
pairs_val, labels_val = create_pairs_balanced(df_val_split, seed=SEED + 7)

valid = np.array([(p[0] in emb_dict) and (p[1] in emb_dict) for p in pairs_val])
pairs_eval  = pairs_val[valid]
labels_eval = labels_val[valid]

scores_val = np.array([
    float(np.dot(emb_dict[p[0]], emb_dict[p[1]]))
    for p in pairs_eval
])
print(f'Scores calculés sur {len(scores_val)} paires')

## 8. Calibration du seuil — FAR fixe

### Différence V9 → V10
- **V9** : seuil Youden (maximise TPR - FPR) → trop permissif pour un dataset très déséquilibré
- **V10** : seuil FAR fixe → on garantit que ≤ TARGET_FAR des paires négatives passent la barre

En biométrie, on préfère manquer quelques vrais matchs plutôt qu'accepter de faux.

In [ ]:
fpr, tpr, thresholds = roc_curve(labels_eval, scores_val)
roc_auc = auc(fpr, tpr)

# ── Seuil Youden (référence V9) ───────────────────────────────────────────
youden_idx = np.argmax(tpr - fpr)
thr_youden = float(thresholds[youden_idx])

# ── Seuil FAR fixe (V10) ─────────────────────────────────────────────────
# On cherche le seuil où FPR ≤ TARGET_FAR
far_candidates = np.where(fpr <= TARGET_FAR)[0]
if len(far_candidates) > 0:
    far_idx = far_candidates[-1]   # le plus haut TPR parmi ceux qui respectent FAR
else:
    far_idx = 0   # fallback
thr_far = float(thresholds[far_idx])
tpr_at_far = float(tpr[far_idx])
fpr_at_far = float(fpr[far_idx])

# ── Correction prior (comme V9) ───────────────────────────────────────────
# Ratio de prior pour ajuster le seuil du dataset déséquilibré
prior_calib = 0.5   # 50/50 dans la calibration
prior_real  = prior_pos
if prior_calib > 0 and prior_real > 0:
    log_ratio = np.log(prior_calib / (1 - prior_calib)) - np.log(prior_real / (1 - prior_real))
    thr_far_corrected = thr_far + 0.5 * log_ratio / (scores_val.std() + 1e-8)
    thr_far_corrected = float(np.clip(thr_far_corrected, thr_far, 0.95))
else:
    thr_far_corrected = thr_far

optimal_threshold = max(thr_far_corrected, THRESHOLD_FLOOR)

print(f'AUC-ROC              : {roc_auc:.4f}')
print(f'Seuil Youden (V9)    : {thr_youden:.4f}')
print(f'Seuil FAR {TARGET_FAR:.3f} (brut) : {thr_far:.4f}  (TPR={tpr_at_far:.3f}, FPR={fpr_at_far:.4f})')
print(f'Seuil FAR corrigé    : {thr_far_corrected:.4f}')
print(f'Seuil final retenu   : {optimal_threshold:.4f}  (floor={THRESHOLD_FLOOR})')

# Rapport de classification
val_preds = (scores_val >= optimal_threshold).astype(int)
print('\nClassification Report (validation) :')
print(classification_report(labels_eval, val_preds, target_names=['Diff.', 'Meme'], zero_division=0))

## 9. Visualisation — Calibration V10

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ROC
axes[0,0].plot(fpr, tpr, label=f'AUC={roc_auc:.4f}')
axes[0,0].plot([0,1],[0,1],'k--')
axes[0,0].scatter(fpr[youden_idx], tpr[youden_idx], color='orange', zorder=5,
                  label=f'Youden (V9)={thr_youden:.3f}')
axes[0,0].scatter(fpr_at_far, tpr_at_far, color='red', zorder=5, marker='*', s=200,
                  label=f'FAR={TARGET_FAR} → {thr_far:.3f}')
axes[0,0].axvline(TARGET_FAR, color='red', linestyle=':', alpha=0.5, label=f'FAR cible={TARGET_FAR}')
axes[0,0].set_title('ROC — V10 vs V9'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# Precision-Recall
prec, rec, thr_pr = precision_recall_curve(labels_eval, scores_val)
pr_auc = auc(rec, prec)
f1s = np.where((prec + rec) > 0, 2*prec*rec/(prec+rec), 0)
axes[0,1].plot(rec, prec, label=f'AP={pr_auc:.4f}')
axes[0,1].set_title('Precision-Recall'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# Distribution
bins = np.linspace(scores_val.min(), scores_val.max(), 80)
axes[1,0].hist(scores_val[labels_eval==1], bins=bins, alpha=0.6, label='Même', color='green', density=True)
axes[1,0].hist(scores_val[labels_eval==0], bins=bins, alpha=0.6, label='Diff.', color='red', density=True)
axes[1,0].axvline(thr_youden, color='orange', linestyle=':', label=f'Youden={thr_youden:.3f}')
axes[1,0].axvline(optimal_threshold, color='blue', linestyle='--', label=f'FAR-seuil={optimal_threshold:.3f}')
mean_pos = scores_val[labels_eval==1].mean()
mean_neg = scores_val[labels_eval==0].mean()
axes[1,0].text(0.05, 0.95,
    f'Séparation = {mean_pos-mean_neg:.3f}\nMoy.pos={mean_pos:.3f}\nMoy.neg={mean_neg:.3f}',
    transform=axes[1,0].transAxes, va='top', fontsize=9,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[1,0].set_title('Distribution (validation)'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# F1 vs seuil
axes[1,1].plot(thr_pr, f1s[:-1], color='purple')
axes[1,1].axvline(optimal_threshold, color='blue', linestyle='--', label=f'Seuil={optimal_threshold:.3f}')
axes[1,1].set_xlabel('Seuil'); axes[1,1].set_ylabel('F1')
axes[1,1].set_title('F1 vs Seuil'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

emb_dim = val_embs.shape[1]
model_name = f'FaceNet+ArcFace (super-{emb_dim}d)' if arcface else 'FaceNet VGGFace2'
plt.suptitle(f'Calibration V10 — {model_name} + MTCNN + TTA {TTA_N_VIEWS} vues', fontsize=13)
plt.tight_layout()
plt.savefig('calibration_v10.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegardé : calibration_v10.png')

## 10. Prédiction sur le Test Set

In [ ]:
test_files, _ = _list_valid_jpgs(TEST_PATH)
test_fps = [str(p) for p in test_files]
test_fns = [p.name for p in test_files]
print(f'Images de test : {len(test_fps)}')

print('Extraction des embeddings test (avec MTCNN)...')
test_embs = get_embeddings_v10(test_fps, BATCH_SIZE)

norms = np.linalg.norm(test_embs, axis=1)
print(f'Normes : min={norms.min():.3f}  max={norms.max():.3f}  (idéal = 1.000)')

# Matrice de similarité vectorisée
print('Calcul matrice de similarité...')
sim_matrix = test_embs @ test_embs.T

utri = np.triu_indices(len(test_fps), k=1)
all_sims = sim_matrix[utri]
print(f'\nDistribution similarités test :')
print(f'  min={all_sims.min():.3f}  médiane={np.median(all_sims):.3f}  max={all_sims.max():.3f}')
for t in [0.45, 0.50, 0.55, 0.60, optimal_threshold, 0.70]:
    n = (all_sims > t).sum()
    print(f'  > {t:.3f} : {n:>7} paires  ({100*n/len(all_sims):.3f}%)')
print(f'\n→ Seuil retenu : {optimal_threshold:.4f}')
print(f'  Paires attendues : ~{n_identites_test_est}')

## 11. Filtrage des matchs

In [ ]:
# Vectorisé : plus de boucle imbriquée Python
rows, cols = utri
sims_utri  = sim_matrix[rows, cols]

mask_dup   = sims_utri > 0.999
mask_match = (sims_utri >= optimal_threshold) & ~mask_dup

matches_df = pd.DataFrame({
    'image1':     [test_fns[i] for i in rows[mask_match]],
    'image2':     [test_fns[j] for j in cols[mask_match]],
    'similarity': np.round(sims_utri[mask_match], 5),
    'is_match':   1,
})

print(f'Quasi-doublons ignorés (sim > 0.999) : {mask_dup.sum()}')
print(f'Paires matchées retenues             : {len(matches_df)}')
print(f'% de paires matchées                 : {100*len(matches_df)/total_test_pairs:.3f}%')
print(f'(attendu : ~{100*prior_pos:.3f}%)')
print(matches_df.head(10))

## 12. Soumission

In [ ]:
matches_df.to_csv('tache1_submission_v10.csv', index=False)

emb_dim = test_embs.shape[1]
print('══════════════════════════════════════════════════════════')
print('Soumission : tache1_submission_v10.csv')
print(f'  Alignement         : MTCNN (160px, margin=20)')
print(f'  Modèle             : FaceNet VGGFace2{" + ArcFace R100" if arcface else ""}')
print(f'  Embedding          : {emb_dim}-dim ({"super-concat" if arcface else "FaceNet seul"})')
print(f'  TTA vues           : {TTA_N_VIEWS}')
print(f'  Calibration        : FAR ≤ {TARGET_FAR} + correction prior')
print(f'  Seuil final        : {optimal_threshold:.4f}')
print(f'  AUC-ROC            : {roc_auc:.4f}')
print(f'  Paires soumises    : {len(matches_df)}')
print(f'  % matchées         : {100*len(matches_df)/total_test_pairs:.3f}%')
print('══════════════════════════════════════════════════════════')

## 13. Visualisation des top matchs

In [ ]:
if len(matches_df) > 0:
    sample = (matches_df[matches_df['similarity'] < 0.999]
              .sort_values('similarity', ascending=False).head(4))
    if len(sample) == 0:
        sample = matches_df.head(4)

    fig, axes = plt.subplots(len(sample), 2, figsize=(8, 4*len(sample)))
    if len(sample) == 1: axes = np.expand_dims(axes, 0)

    for idx, (_, row) in enumerate(sample.iterrows()):
        for col, img_name in enumerate([row['image1'], row['image2']]):
            img = cv2.imread(str(TEST_PATH / img_name))
            if img is not None:
                axes[idx, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            label = img_name if col == 0 else f"{img_name}\nSim: {row['similarity']:.3f}"
            axes[idx, col].set_title(label, fontsize=9)
            axes[idx, col].axis('off')

    plt.suptitle('Top Paires Matchées — V10 (MTCNN + FAR)', fontsize=13)
    plt.tight_layout()
    plt.savefig('matches_v10.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Sauvegardé : matches_v10.png')
else:
    print('Aucune paire — essaie de baisser THRESHOLD_FLOOR.')